In [1]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import random


In [2]:
data_path = Path("../data/raw/ESConv.json")
with open(data_path, encoding="utf-8") as f:
    raw = json.load(f)

print(type(raw))
print(len(raw))

<class 'list'>
1300


In [3]:
print(json.dumps(raw[0], indent=2, ensure_ascii=False))

{
  "experience_type": "Previous Experience",
  "emotion_type": "anxiety",
  "problem_type": "job crisis",
  "situation": "I hate my job but I am scared to quit and seek a new career.",
  "survey_score": {
    "seeker": {
      "initial_emotion_intensity": "5",
      "empathy": "5",
      "relevance": "5",
      "final_emotion_intensity": "1"
    },
    "supporter": {
      "relevance": "5"
    }
  },
  "dialog": [
    {
      "speaker": "seeker",
      "annotation": {},
      "content": "Hello\n"
    },
    {
      "speaker": "supporter",
      "annotation": {
        "strategy": "Question"
      },
      "content": "Hello, what would you like to talk about?"
    },
    {
      "speaker": "seeker",
      "annotation": {},
      "content": "I am having a lot of anxiety about quitting my current job. It is too stressful but pays well\n"
    },
    {
      "speaker": "supporter",
      "annotation": {
        "strategy": "Question"
      },
      "content": "What makes your job stressful

In [4]:
print(json.dumps(raw[100], indent=2, ensure_ascii=False))

{
  "experience_type": "Previous Experience",
  "emotion_type": "anxiety",
  "problem_type": "breakup with partner",
  "situation": "My girl has decided that Brad from accounting is someone that she has secretly wanted for a long time. And now that he is divorced, she is breaking up with me to get with him.",
  "survey_score": {
    "seeker": {
      "initial_emotion_intensity": "3",
      "empathy": "3",
      "relevance": "4",
      "final_emotion_intensity": "2"
    },
    "supporter": {}
  },
  "dialog": [
    {
      "speaker": "seeker",
      "annotation": {},
      "content": "I cannot believe my Girl is breaking up with me...."
    },
    {
      "speaker": "seeker",
      "annotation": {},
      "content": "I am very anxious about how to cope with my emotions."
    },
    {
      "speaker": "seeker",
      "annotation": {},
      "content": "Can you IMAGINE?? Breaking up with ME?"
    },
    {
      "speaker": "supporter",
      "annotation": {
        "strategy": "Providing S

In [5]:
strategy_counts = Counter()
for conv in raw:
    for turn in conv["dialog"]:
        if turn["speaker"] == "supporter":
            strategy = turn["annotation"].get("strategy")
            if strategy is not None:
                strategy_counts[strategy] += 1

In [6]:
strategy_df = (
    pd.Series(strategy_counts)
    .sort_values(ascending=False)
    .rename_axis("strategy")
    .reset_index(name="count")
)
strategy_df["pct"] = 100 * strategy_df["count"] / strategy_df["count"].sum()
print(strategy_df)

                      strategy  count        pct
0                     Question   3801  20.684589
1                       Others   3341  18.181323
2        Providing Suggestions   2954  16.075316
3  Affirmation and Reassurance   2827  15.384197
4              Self-disclosure   1713   9.321942
5       Reflection of feelings   1436   7.814541
6                  Information   1215   6.611885
7  Restatement or Paraphrasing   1089   5.926208


In [7]:
problem_type_counts = Counter(conv["problem_type"] for conv in raw)
for label, count in problem_type_counts.most_common():
    print(f"{label!r}: {count}")

print(f"\nTotal distinct problem types: {len(problem_type_counts)}")

emotion_counts = Counter(conv["emotion_type"] for conv in raw)
for label, count in emotion_counts.most_common():
    print(f"{label!r}: {count}")
print(f"\nTotal distinct emotions: {len(emotion_counts)}")

experience_type_counts = Counter(conv["experience_type"] for conv in raw)
for label, count in experience_type_counts.most_common():
    print(f"{label!r}: {count}")
print(f"\nTotal distinct experience types: {len(experience_type_counts)}")


'ongoing depression': 351
'job crisis': 280
'breakup with partner': 239
'problems with friends': 179
'academic pressure': 156
'Sleep Problems': 28
'Procrastination': 13
'Alcohol Abuse': 12
'Appearance Anxiety': 12
'conflict with parents': 10
'Issues with Children': 10
'Issues with Parents': 8
'School Bullying': 2

Total distinct problem types: 13
'anxiety': 354
'depression': 334
'sadness': 308
'anger': 111
'fear': 95
'shame': 42
'disgust': 40
'nervousness': 13
'pain': 1
'jealousy': 1
'guilt': 1

Total distinct emotions: 11
'Current Experience': 991
'Previous Experience': 309

Total distinct experience types: 2


In [8]:
def sample_conversations_for_problem_type(problem_type, n=5):
    matches = [conv for conv in raw if conv["problem_type"] == problem_type]
    print(f"{problem_type}: {len(matches)} conversations total")
    for conv in matches[:n]:
        print(f"\n--- situation ---\n{conv['situation']}")

sample_conversations_for_problem_type("conflict with parents")
print("\n\n")
sample_conversations_for_problem_type("Issues with Parents")

conflict with parents: 10 conversations total

--- situation ---
I am getting ready to sue my mother for not believing my step dad raped me.

--- situation ---
I am afraid of how my parents will react when I tell them that I am not going to attend the holidays

--- situation ---
My parents are always telling me I should be disciplining my children differently.  Things are different than when they raised us.

--- situation ---
I am fighting with my parents because I am 17 and have just began my first relationship and my parents have grounded me after finding out that I am dating before the age of 18

--- situation ---
I love my parents but they want me to pursue business major while I am doing a psychology major to be Physician Assistant one day. They feel I am letting them down but in reality I want to be in the medicine field. I do not know what to do



Issues with Parents: 8 conversations total

--- situation ---
My parents have this expectation of me to give back to them even thoug

In [9]:
convo_lengths = [len(conv["dialog"]) for conv in raw]
utterance_lengths = [len(turn["content"].split()) for conv in raw for turn in conv["dialog"]]

print("Turns per conversation:")
print(pd.Series(convo_lengths).describe())

print("\nUtterance word count:")
print(pd.Series(utterance_lengths).describe())

Turns per conversation:
count    1300.000000
mean       29.511538
std        10.156846
min        16.000000
25%        23.000000
50%        27.000000
75%        33.000000
max       120.000000
dtype: float64

Utterance word count:
count    38365.000000
mean        16.396716
std         13.306965
min          1.000000
25%          7.000000
50%         13.000000
75%         22.000000
max        166.000000
dtype: float64


In [10]:
random.seed(42)

def sample_turns_for_strategy(strategy, n=5):
    matches = [
        (conv["problem_type"], turn["content"])
        for conv in raw
        for turn in conv["dialog"]
        if turn["speaker"] == "supporter" and turn["annotation"].get("strategy") == strategy
    ]
    return random.sample(matches, min(n, len(matches)))

all_strategies = sorted(strategy_counts.keys())

for strategy in all_strategies:
    print(f"\n=== {strategy} ===")
    for problem_type, content in sample_turns_for_strategy(strategy):
        print(f"[{problem_type}] {content}")


=== Affirmation and Reassurance ===
[job crisis] I am more than happy to talk any  time! Try not to worry too much.
[breakup with partner] I wish I could fix this, but I can't. The road will be rocky and I don't know how I would have made it without people to encourage and support me, so I'm glad a friend is willing to have you stay with them until you can get on your feet. 
[problems with friends] I hope I have been able to help you a little and set your mind at rest. I wish you a very happy New Year! 
[breakup with partner] That is wonderful news!  See you are starting to make a plan for a better life for you and your children.  You so got this!
[job crisis] I know it is a super tough time, but I know you can make it through it!


=== Information ===
[job crisis] I see. well the great thing about accountants is that every business needs them, regardless of the economy
[problems with friends] Bad things inevitably happen to terrible people. its usually just a matter of time. 
[job cr

In [11]:
long_convos = (pd.Series(convo_lengths) > 60).sum()
print(f"Conversations with >60 turns: {long_convos} of {len(convo_lengths)}")

supporter_lengths = [len(turn["content"].split()) for conv in raw for turn in conv["dialog"] if turn["speaker"] == "supporter"]
long_supporter_turns = (pd.Series(supporter_lengths) > 40).sum()
print(f"Supporter turns >40 words: {long_supporter_turns} of {len(supporter_lengths)}")

Conversations with >60 turns: 25 of 1300
Supporter turns >40 words: 1190 of 18376


In [12]:
"""
Check whether each strategy clusters at a particular point in the conversation (normalized 0=start, 1=end).
"""
rows = []
for conv in raw:
    n_turns = len(conv["dialog"])
    for i, turn in enumerate(conv["dialog"]):
        if turn["speaker"] == "supporter":
            strategy = turn["annotation"].get("strategy")
            if strategy is not None:
                normalized_position = i / (n_turns - 1)
                rows.append({"strategy": strategy, "normalized_position": normalized_position})

position_df = pd.DataFrame(rows)

position_stats = (
    position_df.groupby("strategy")["normalized_position"]
    .agg(["mean", "std", "count"])
    .sort_values("mean")
)
print(position_stats)

                                 mean       std  count
strategy                                              
Question                     0.288800  0.272722   3801
Restatement or Paraphrasing  0.331173  0.253553   1089
Reflection of feelings       0.480819  0.255128   1436
Self-disclosure              0.510338  0.243165   1713
Affirmation and Reassurance  0.538830  0.271427   2827
Providing Suggestions        0.591559  0.217294   2954
Information                  0.599256  0.232928   1215
Others                       0.663569  0.335661   3341


In [13]:
"""
Compare each strategy's share of conversation-ending supporter turns to its overall share,
to see which strategies disproportionately close conversations.
"""

last_supporter_strategy = []
for conv in raw:
    supporter_turns = [t for t in conv["dialog"] if t["speaker"] == "supporter"]
    if supporter_turns:
        last_supporter_strategy.append(supporter_turns[-1]["annotation"].get("strategy"))

last_strategy_counts = pd.Series(last_supporter_strategy).value_counts()
last_strategy_pct = 100 * last_strategy_counts / last_strategy_counts.sum()

comparison = pd.DataFrame({
    "last_turn_pct": last_strategy_pct,
    "overall_pct": 100 * pd.Series(strategy_counts) / sum(strategy_counts.values()),
}).sort_values("last_turn_pct", ascending=False)

print(comparison)

                             last_turn_pct  overall_pct
Others                           58.769231    18.181323
Affirmation and Reassurance      10.769231    15.384197
Providing Suggestions            10.230769    16.075316
Information                       5.846154     6.611885
Reflection of feelings            4.384615     7.814541
Question                          4.076923    20.684589
Self-disclosure                   3.615385     9.321942
Restatement or Paraphrasing       2.307692     5.926208


In [14]:
"""
Check whether "Others" turns are lexically greetings/closings, split by opening vs. closing phrases, 
to see if "Others" is a conversational-bookend strategy rather than a closing-only one.
"""

opening_words = [
    "hello", "hi there", "hi,", "hi!", "hey there", "hi ",
    "how are you", "how's it going", "what would you like to talk about",
    "what's on your mind", "what brings you", "how can i help",
    "what seems to be", "tell me what's going on", "nice to meet you",
]

closing_words = [
    "bye", "goodbye", "good bye", "take care", "thank", "thanks",
    "welcome", "glad", "hope",
    "good luck", "best of luck", "merry christmas", "happy holidays",
    "happy new year", "talk to you", "reach out", "wish you",
    "all the best", "take it easy", "stay strong", "you got this",
    "have a good", "have a nice", "have a great", "have a wonderful",
]

def contains_any(text, phrases):
    text_lower = text.lower()
    return any(p in text_lower for p in phrases)

others_turns = [
    turn["content"] for conv in raw for turn in conv["dialog"]
    if turn["speaker"] == "supporter" and turn["annotation"].get("strategy") == "Others"
]

opening_matches = sum(contains_any(t, opening_words) for t in others_turns)
closing_matches = sum(contains_any(t, closing_words) for t in others_turns)
either_matches = sum(contains_any(t, opening_words) or contains_any(t, closing_words) for t in others_turns)

print(f"Opening-phrase matches: {opening_matches} of {len(others_turns)} ({100*opening_matches/len(others_turns):.1f}%)")
print(f"Closing-phrase matches: {closing_matches} of {len(others_turns)} ({100*closing_matches/len(others_turns):.1f}%)")
print(f"Either: {either_matches} of {len(others_turns)} ({100*either_matches/len(others_turns):.1f}%)")

Opening-phrase matches: 323 of 3341 (9.7%)
Closing-phrase matches: 1200 of 3341 (35.9%)
Either: 1510 of 3341 (45.2%)


In [15]:
# BUGFIX: this cell was missing -- cell 14 (below) referenced
# first_strategy_counts/first_supporter_strategy/first_others_pct/last_others_pct
# without ever computing them in this notebook. The saved outputs below were
# real (they match the ~33%% figure cited in preprocessing.py), but as saved
# the notebook could not be reproduced by Restart & Run All -- fixed by
# restoring the computation, mirroring the last_supporter_strategy pattern
# already used two cells above.
first_supporter_strategy = []
for conv in raw:
    supporter_turns = [t for t in conv["dialog"] if t["speaker"] == "supporter"]
    if supporter_turns:
        first_supporter_strategy.append(supporter_turns[0]["annotation"].get("strategy"))

first_strategy_counts = pd.Series(first_supporter_strategy).value_counts()
first_strategy_pct = 100 * first_strategy_counts / first_strategy_counts.sum()
first_others_pct = first_strategy_pct.get("Others", 0.0)
last_others_pct = last_strategy_pct.get("Others", 0.0)


In [16]:
others_first_count = first_strategy_counts.get("Others", 0)
others_last_count = last_strategy_counts.get("Others", 0)
total_others = strategy_counts["Others"]

pct_others_that_are_opening = 100 * others_first_count / total_others
pct_others_that_are_closing = 100 * others_last_count / total_others

print(f"Out of {len(first_supporter_strategy)} opening turns, {first_others_pct:.1f}% are 'Others'")
print(f"Out of {len(last_supporter_strategy)} closing turns, {last_others_pct:.1f}% are 'Others'")
print()
print(f"Out of {total_others} 'Others' turns, {pct_others_that_are_opening:.1f}% are the conversation's opening turn")
print(f"Out of {total_others} 'Others' turns, {pct_others_that_are_closing:.1f}% are the conversation's closing turn")

Out of 1300 opening turns, 25.5% are 'Others'
Out of 1300 closing turns, 58.8% are 'Others'

Out of 3341 'Others' turns, 9.9% are the conversation's opening turn
Out of 3341 'Others' turns, 22.9% are the conversation's closing turn


In [17]:
def count_consecutive_same_speaker(dialog):
    return sum(
        1 for i in range(1, len(dialog))
        if dialog[i]["speaker"] == dialog[i-1]["speaker"]
    )

consecutive_counts = [count_consecutive_same_speaker(conv["dialog"]) for conv in raw]
convos_with_any_consecutive = sum(1 for c in consecutive_counts if c > 0)
print(f"Conversations with at least one same-speaker-in-a-row: {convos_with_any_consecutive} of {len(raw)}")
print(f"Total same-speaker-adjacent-turn events: {sum(consecutive_counts)}")

first_speakers = Counter(conv["dialog"][0]["speaker"] for conv in raw)
print(f"First-turn speaker: {first_speakers}")

speaker_values = Counter(turn["speaker"] for conv in raw for turn in conv["dialog"])
print(f"All speaker values: {speaker_values}")

Conversations with at least one same-speaker-in-a-row: 1178 of 1300
Total same-speaker-adjacent-turn events: 7973
First-turn speaker: Counter({'supporter': 683, 'seeker': 617})
All speaker values: Counter({'seeker': 19989, 'supporter': 18376})


## Exploration conclusions

Every supporter turn carries a `strategy` label — 0 missing across 18,376 turns.
8 distinct raw strategy strings, no casing/spelling collisions.

13 distinct `problem_type` strings vs. 12 named in the README. `conflict with
parents` (10) and `Issues with Parents` (8) overlap in content with no categorical
distinction (e.g. both contain a "grounded by parents" situation) — merged into
`Issues with Parents` in preprocessing.

Turns per conversation: 16–120 (mean 29.5, median 27); min=16 for every conversation.
25/1,300 (1.9%) exceed 60 turns.

Utterance length: 1–166 words (mean 16.4, median 13). 1,190/18,376 supporter turns
(6.5%) exceed 40 words — the GenAI subtask's response cap — so even a perfect
generator cannot match gold length on this subset.

Turns are not alternating: 1,178/1,300 conversations (90.6%) contain ≥1 same-speaker-
in-a-row event (7,973 events total). 683/1,300 (52.5%) open with a supporter turn,
meaning over half the first-turn predictions in the dataset have no seeker turn
preceding them.

Sampled examples: Self-disclosure turns build rapport and sometimes end in a
follow-up question. Information is mixed — some factual, some reassurance framed as
fact; one sampled item ("Bye, have a wonderful day.") looks mislabeled. "Others" is
not semantically coherent — session-timeout text, holiday closings, check-ins, and
bare acknowledgments all appear under it.

## Open decisions

1. Context window: full history vs. fixed window (size + justification); handling
   for the 683 supporter-turn-first conversations; speaker-role tagging; whether to
   include `problem_type`/`emotion_type` as model input.
2. "Others": force into one of 3 coarse classes, or document as a known misfit.
3. 8→3 mapping: Self-disclosure and Information not yet assigned — sampled evidence
   is ambiguous for both.
4. `conflict with parents` vs. `Issues with Parents`: merge or keep separate —
   pending a content read, not just label similarity.
5. Architecture: pick one, justify against data size/imbalance/context length. No
   comparative testing (last submission's failure mode).
6. Split ratios/stratification: problem-type categories as small as n=2 constrain
   options.
7. Split protocol: conversation-level vs. random-utterance gap — required number,
   not yet measured.